# 04 · Claude Agent SDK：Anthropic 原生工具调用

不依赖任何框架，直接用 Anthropic Python SDK 实现 Agent——理解工具调用的底层机制，写最可控的 Agent 代码。

**你将学到：**

| 章节 | 内容 |
|------|------|
| 第一章 | 工具调用协议：tool_use / tool_result 消息格式 |
| 第二章 | 单轮工具调用实战 |
| 第三章 | 多轮工具调用：手写 Agent Loop |
| 第四章 | 流式输出 |
| 第五章 | 子 Agent 模式 |
| 第六章 | 长对话管理与 Token 优化 |
| 小结 | Claude SDK vs LangChain vs AutoGen |


## 第一章：工具调用协议

Anthropic 的工具调用是一个「请求-执行-返回」的循环，消息格式如下：

```mermaid
sequenceDiagram
    participant U as 你的代码
    participant C as Claude
    U->>C: messages + tools 定义
    C->>U: stop_reason=tool_use, content=[tool_use block]
    Note over U: 执行工具函数
    U->>C: messages + tool_result block
    C->>U: stop_reason=end_turn, 最终回复
```

关键消息类型：

| stop_reason | 含义 | 你需要做什么 |
|-------------|------|--------------|
| `tool_use` | 模型要调用工具 | 执行工具，返回 `tool_result` |
| `end_turn` | 模型完成回复 | 提取 `text` 内容即可 |
| `max_tokens` | token 超限 | 增加 `max_tokens` 或截断上下文 |


In [ ]:
import os, json
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
MODEL = 'claude-sonnet-4-6'
print(f'Anthropic SDK 版本: {anthropic.__version__}')


## 第二章：单轮工具调用实战

定义一个计算器工具，观察完整的请求/响应格式：

In [ ]:
# 定义工具 Schema
tools = [{
    'name': 'calculator',
    'description': 'Evaluate a mathematical expression. Supports +, -, *, /, **, sqrt, etc.',
    'input_schema': {
        'type': 'object',
        'properties': {
            'expression': {'type': 'string', 'description': 'Math expression to evaluate, e.g. "2**32" or "sqrt(144)"'}
        },
        'required': ['expression']
    }
}]

import math

def run_calculator(expression: str) -> str:
    try:
        result = eval(expression, {'__builtins__': {}},
                     {'sqrt': math.sqrt, 'pi': math.pi, 'log': math.log, 'sin': math.sin, 'cos': math.cos})
        return str(result)
    except Exception as e:
        return f'Error: {e}'

# 第一轮：发送问题 + 工具定义
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=[{'role': 'user', 'content': 'What is 2 to the power of 32, and what is the square root of that result?'}]
)

print(f'stop_reason: {response.stop_reason}')
print(f'content blocks: {len(response.content)}')
for block in response.content:
    print(f'  type={block.type}', end='')
    if block.type == 'tool_use':
        print(f'  name={block.name}  input={block.input}')
    else:
        print(f'  text={getattr(block, "text", "")[:60]}')


## 第三章：多轮工具调用——手写 Agent Loop

核心逻辑：循环调用 API，遇到 `tool_use` 就执行工具并返回结果，直到 `end_turn`：

In [ ]:
def run_agent(user_message: str, tools: list, tool_funcs: dict, max_steps: int = 10) -> str:
    """手写 Agent Loop：自动处理多轮工具调用直到 end_turn"""
    messages = [{'role': 'user', 'content': user_message}]

    for step in range(max_steps):
        response = client.messages.create(
            model=MODEL, max_tokens=1024, tools=tools, messages=messages
        )
        print(f'[Step {step+1}] stop_reason={response.stop_reason}')

        if response.stop_reason == 'end_turn':
            # 提取最终文本回复
            return next((b.text for b in response.content if b.type == 'text'), '')

        if response.stop_reason == 'tool_use':
            # 把模型回复加入消息历史
            messages.append({'role': 'assistant', 'content': response.content})
            # 执行所有工具请求，收集结果
            tool_results = []
            for block in response.content:
                if block.type == 'tool_use':
                    fn = tool_funcs.get(block.name)
                    result = fn(**block.input) if fn else f'Tool {block.name} not found'
                    print(f'  -> {block.name}({block.input}) = {result}')
                    tool_results.append({
                        'type': 'tool_result',
                        'tool_use_id': block.id,
                        'content': result
                    })
            messages.append({'role': 'user', 'content': tool_results})
        else:
            break

    return 'Max steps reached'

# 测试：需要两步工具调用
answer = run_agent(
    'What is 2^32? Then what is the square root of that number? Give me both results.',
    tools=tools,
    tool_funcs={'calculator': run_calculator}
)
print(f'\n最终回答: {answer}')


## 第四章：流式输出

`stream=True` 让输出逐 token 显示，工具调用也可以流式处理：

In [ ]:
import sys

def stream_agent(user_message: str) -> str:
    """流式输出 + 工具调用"""
    messages = [{'role': 'user', 'content': user_message}]
    full_response = ''

    with client.messages.stream(
        model=MODEL, max_tokens=512, tools=tools, messages=messages
    ) as stream:
        for event in stream:
            if hasattr(event, 'type'):
                if event.type == 'content_block_delta':
                    delta = event.delta
                    if hasattr(delta, 'text'):
                        print(delta.text, end='', flush=True)
                        full_response += delta.text
        final = stream.get_final_message()

    print()  # 换行

    # 如果模型要调用工具，正常处理
    if final.stop_reason == 'tool_use':
        print('[工具调用中...]')
        messages.append({'role': 'assistant', 'content': final.content})
        tool_results = []
        for block in final.content:
            if block.type == 'tool_use':
                result = run_calculator(**block.input)
                tool_results.append({'type': 'tool_result', 'tool_use_id': block.id, 'content': result})
        messages.append({'role': 'user', 'content': tool_results})
        # 第二轮获取最终回答
        follow_up = client.messages.create(model=MODEL, max_tokens=512, tools=tools, messages=messages)
        full_response = next((b.text for b in follow_up.content if b.type == 'text'), '')
        print(full_response)

    return full_response

stream_agent('Calculate the area of a circle with radius 5, using pi = 3.14159.')


## 第五章：子 Agent 模式

Orchestrator 把大任务分解给专门化的 Subagent，最后汇总结果：


In [ ]:
def create_subagent(role: str, task: str) -> str:
    """一个专门化的 Subagent，完成特定任务"""
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=f'You are a {role}. Be concise.',
        messages=[{'role': 'user', 'content': task}]
    )
    return response.content[0].text

def orchestrator(topic: str) -> dict:
    """Orchestrator：并行调度多个 Subagent"""
    print(f'分析主题: {topic}\n')

    # 三个专门化子任务（实际生产中可并行执行）
    facts = create_subagent('fact researcher', f'List 3 key facts about: {topic}')
    print(f'[Researcher]: {facts[:100]}...')

    pros_cons = create_subagent('critical analyst', f'List 2 pros and 2 cons of: {topic}')
    print(f'[Analyst]: {pros_cons[:100]}...')

    summary = create_subagent('technical writer',
        f'Given these facts: {facts}\nAnd analysis: {pros_cons}\nWrite a 50-word summary about: {topic}')

    return {'facts': facts, 'pros_cons': pros_cons, 'summary': summary}

result = orchestrator('using LLMs in production applications')
print(f'\n最终摘要:\n{result["summary"]}')


## 第六章：长对话管理与 Token 优化

当对话超出上下文窗口时，用摘要压缩策略保留语义：

In [ ]:
def compress_history(messages: list, keep_last: int = 4) -> list:
    """当消息超过阈值时，把早期消息压缩为摘要"""
    if len(messages) <= keep_last + 2:  # +2 for system overhead
        return messages

    # 把早期消息摘要化
    old_messages = messages[:-keep_last]
    recent_messages = messages[-keep_last:]

    old_text = '\n'.join([
        f"{m['role']}: {m['content'] if isinstance(m['content'], str) else str(m['content'])[:100]}"
        for m in old_messages
    ])

    summary_response = client.messages.create(
        model=MODEL, max_tokens=200,
        messages=[{'role': 'user', 'content': f'Summarize this conversation in 2-3 sentences:\n{old_text}'}]
    )
    summary = summary_response.content[0].text

    compressed = [{'role': 'user', 'content': f'[Previous conversation summary]: {summary}'}] + recent_messages
    print(f'压缩: {len(messages)} 条消息 → {len(compressed)} 条 (保留语义摘要)')
    return compressed

# 演示：模拟一段对话历史并压缩
mock_history = [
    {'role': 'user', 'content': 'What is machine learning?'},
    {'role': 'assistant', 'content': 'Machine learning is a subset of AI...'},
    {'role': 'user', 'content': 'What about deep learning?'},
    {'role': 'assistant', 'content': 'Deep learning uses neural networks...'},
    {'role': 'user', 'content': 'How does backpropagation work?'},
    {'role': 'assistant', 'content': 'Backpropagation calculates gradients...'},
]

compressed = compress_history(mock_history, keep_last=2)
for m in compressed:
    print(f'  [{m["role"]}]: {m["content"][:80]}...')


## 小结：Claude SDK vs LangChain vs AutoGen

```mermaid
flowchart LR
    NEED[我的需求] --> Q1{复杂度}
    Q1 -->|简单工具调用| SDK[Claude SDK\n最轻量，最可控]
    Q1 -->|需要 RAG/Chain| LC[LangChain\n丰富组件生态]
    Q1 -->|需要多 Agent 协作| AG[AutoGen\n角色分工对话]
    SDK -->|需要数据检索| LC
    LC & AG -->|底层都可以| SDK
```

| 维度 | Claude SDK | LangChain | AutoGen |
|------|-----------|-----------|----------|
| **学习曲线** | 低 | 中 | 中 |
| **灵活性** | 最高 | 高 | 中 |
| **工具生态** | 自己实现 | 丰富 | 中等 |
| **多 Agent** | 需手写 | 有限支持 | 核心能力 |
| **适合** | 生产精细控制 | 快速原型 | 复杂协作任务 |

**建议**：先学 Claude SDK 理解底层原理，再根据需求选用 LangChain 或 AutoGen 提效。
